<a href="https://colab.research.google.com/github/kethelineberlin/Intelig-ncia-Computacional-Aplicada-Gera-o-de-Energia-e-Sustentabilidade/blob/main/PROTOCOLO_EXPERIMENTAL_AVALIA%C3%87%C3%83O_DE_MODELOS_DE_DEEP_LEARNING_(MHP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Passo 1.1 - Definir os conjuntos de variáveis (Feature Sets): organizar e documentar a justificativa para 4 grupos de variáveis.

FS-1 (Full Set) terá todas as variáveis.
- Variáveis Incluídas (8 variáveis)  clearSkyGhi  
fitLuminanceCSI  
whitePixelRatio  
sunLuminance  
cloudsCoverage  
cloudsMovement1  
cloudsMovement5  
cloudsMovement15  

O FS-1 serve como o cenário de referência (baseline) absoluto para a pesquisa. Ele contém a totalidade das variáveis originais do Método Híbrido de Predição (MHP). A justificativa central para mantê-lo e testá-lo é estabelecer os parâmetros máximos de erro e acerto. Como o objetivo do estudo é provar que a seleção de características guiada por Explainable AI (XAI) pode otimizar as redes neurais sem grande perda de precisão, é cientificamente necessário ter os resultados do conjunto completo (FS-1) para poder quantificar e validar (através das métricas RMSE, MAPE e R²) se os modelos mais enxutos são de fato mais eficientes do que o modelo alimentado com todas as variáveis climáticas e de movimentação de nuvens.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import zipfile
import os

# 1. Carregar o dataset localmente (após fazer o upload no painel lateral)
zip_path = '/content/features_image_mhp.zip'
extraction_path = '/content/extracted_data'

print("Descompactando o arquivo ZIP...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)
print(f"Arquivo ZIP descompactado para: {extraction_path}")

# Encontrar o arquivo CSV dentro da pasta descompactada
caminho_arquivo_csv = os.path.join(extraction_path, 'features_image_mhp.csv')

print("Carregando o dataset local...")
df = pd.read_csv(caminho_arquivo_csv, delimiter=';', decimal=',')

# Limpar os nomes das colunas para evitar erros com espaços em branco ocultos
df.columns = df.columns.str.strip()

# 2. Definir o conjunto de variáveis FS-1 (Full Set)
fs1_features = [
    'clearSkyGhi',
    'fitLuminanceCSI',
    'whitePixelRatio',
    'sunLuminance',
    'cloudsCoverage',
    'cloudsMovement1',
    'cloudsMovement5',
    'cloudsMovement15'
]

# ==========================================
# TRECHO CORRIGIDO
# ==========================================
# Definir a variável alvo (target)
target_col = 'clearSkyGhi'

# Juntar features e target, mas remover qualquer duplicata usando dict.fromkeys()
cols_to_use = list(dict.fromkeys(fs1_features + [target_col]))

# Filtrar o dataframe apenas com as colunas necessárias e remover linhas com dados faltantes (NaN)
df_fs1 = df[cols_to_use].dropna()

print(f"Dataset filtrado com sucesso. Total de registros: {len(df_fs1)}")

# 3. Aplicar o MinMaxScaler
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(df_fs1[fs1_features])
# O reshape garante que o y saia com o formato correto (X, 1) em vez de (X, 2)
y_scaled = scaler_y.fit_transform(df_fs1[target_col].values.reshape(-1, 1))
# ==========================================
# FIM DO TRECHO CORRIGIDO
# ==========================================

# 4. Função para gerar as Janelas Deslizantes
def create_sliding_windows(X_data, y_data, lookback_window, horizon):
    """
    X_data: features normalizadas
    y_data: target normalizado
    lookback_window: tamanho da janela de observação (quantos passos no tempo o modelo vai "olhar" para trás)
    horizon: horizonte de predição (1, 5 ou 15 minutos)
    """
    X_windows, y_windows = [], []

    # O loop garante que não vamos acessar índices fora do tamanho do array
    for i in range(len(X_data) - lookback_window - horizon + 1):
        X_windows.append(X_data[i : (i + lookback_window)])
        # O target é o valor no momento (i + lookback_window + horizon - 1)
        y_windows.append(y_data[i + lookback_window + horizon - 1])

    return np.array(X_windows), np.array(y_windows)

# 5. Criando os datasets finais para os experimentos
lookback = 60

print("\nGerando janelas deslizantes...")
X_1min, y_1min = create_sliding_windows(X_scaled, y_scaled, lookback_window=lookback, horizon=1)
X_5min, y_5min = create_sliding_windows(X_scaled, y_scaled, lookback_window=lookback, horizon=5)
X_15min, y_15min = create_sliding_windows(X_scaled, y_scaled, lookback_window=lookback, horizon=15)

print("\nPreparação do FS-1 concluída!")
print(f"Formato final (Shape) para horizonte de 1 min:  X={X_1min.shape}, y={y_1min.shape}")
print(f"Formato final (Shape) para horizonte de 5 min:  X={X_5min.shape}, y={y_5min.shape}")
print(f"Formato final (Shape) para horizonte de 15 min: X={X_15min.shape}, y={y_15min.shape}")

Descompactando o arquivo ZIP...
Arquivo ZIP descompactado para: /content/extracted_data
Carregando o dataset local...
Dataset filtrado com sucesso. Total de registros: 764867

Gerando janelas deslizantes...

Preparação do FS-1 concluída!
Formato final (Shape) para horizonte de 1 min:  X=(764807, 60, 8), y=(764807, 1)
Formato final (Shape) para horizonte de 5 min:  X=(764803, 60, 8), y=(764803, 1)
Formato final (Shape) para horizonte de 15 min: X=(764793, 60, 8), y=(764793, 1)


Validação do Pré-processamento (Conjunto FS-1)

Os tensores foram gerados com sucesso e estão no formato adequado para alimentar as redes neurais:

Formato das Entradas (X): Estrutura tridimensional (ex: X=(764807, 60, 8)), que é o padrão exigido por arquiteturas como LSTM e CNN-1D. Cada pacote de treino contém 60 passos no tempo cruzando as 8 variáveis selecionadas para o FS-1.

Isolamento do Alvo (y): O shape final (..., 1) confirma que a variável a ser prevista foi separada corretamente, garantindo que não haja vazamento de dados (data leakage) durante o treinamento.

Comportamento dos Horizontes: Há uma leve redução no total de amostras conforme o horizonte de previsão aumenta (de 1 min para 5 e 15 min). Isso é matematicamente esperado na técnica de janelas deslizantes, pois a projeção futura atinge o limite final da série temporal mais cedo, descartando os últimos registros.

FS-2: Top-3 XAI (Variáveis de Maior Importância)Variáveis Incluídas (3 variáveis):  
clearSkyGhi  
fitLuminanceCSI  
whitePixelRatio

O conjunto FS-2 é fundamentado diretamente nos resultados da análise de Inteligência Artificial Explicável (XAI), utilizando as técnicas de Permutation Importance e SHAP aplicadas ao modelo MLP original. O diagnóstico prévio revelou que a capacidade preditiva não está distribuída uniformemente entre todas as variáveis, sendo que apenas três delas concentram cerca de 58% de toda a importância do modelo. Sendo elas: clearSkyGhi (com aproximadamente 34% de importância), fitLuminanceCSI (~15%) e whitePixelRatio (~9%). A adoção deste subconjunto visa testar a hipótese de que modelos de Deep Learning (LSTM, GRU e CNN-1D) podem atingir uma acurácia comparável ou até superior operando apenas com a informação mais essencial, reduzindo drasticamente a dimensionalidade dos dados e o custo computacional do Método Híbrido de Predição (MHP).

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# 1. Carregar o arquivo CSV já descompactado no passo anterior
caminho_arquivo_csv = '/content/extracted_data/features_image_mhp.csv'

print("Carregando o dataset local para o FS-2...")
df = pd.read_csv(caminho_arquivo_csv, delimiter=';', decimal=',')

# Limpar os nomes das colunas
df.columns = df.columns.str.strip()

# 2. Definir o conjunto de variáveis FS-2 (Top-3 XAI)
fs2_features = [
    'clearSkyGhi',
    'fitLuminanceCSI',
    'whitePixelRatio'
]

# Definir a variável alvo (target)
target_col = 'clearSkyGhi'

# Juntar features e target, removendo duplicatas
cols_to_use = list(dict.fromkeys(fs2_features + [target_col]))

# Filtrar o dataframe e remover NaNs
df_fs2 = df[cols_to_use].dropna()

print(f"Dataset do FS-2 filtrado com sucesso. Total de registros: {len(df_fs2)}")

# 3. Aplicar o MinMaxScaler exclusivo para o FS-2
scaler_X_fs2 = MinMaxScaler()
scaler_y_fs2 = MinMaxScaler()

X_scaled_fs2 = scaler_X_fs2.fit_transform(df_fs2[fs2_features])
y_scaled_fs2 = scaler_y_fs2.fit_transform(df_fs2[target_col].values.reshape(-1, 1))

# 4. Função para gerar as Janelas Deslizantes (mesma estrutura)
def create_sliding_windows(X_data, y_data, lookback_window, horizon):
    X_windows, y_windows = [], []
    for i in range(len(X_data) - lookback_window - horizon + 1):
        X_windows.append(X_data[i : (i + lookback_window)])
        y_windows.append(y_data[i + lookback_window + horizon - 1])
    return np.array(X_windows), np.array(y_windows)

# 5. Criando os datasets finais para o FS-2
lookback = 60

print("\nGerando janelas deslizantes para o FS-2...")
X_1min_fs2, y_1min_fs2 = create_sliding_windows(X_scaled_fs2, y_scaled_fs2, lookback_window=lookback, horizon=1)
X_5min_fs2, y_5min_fs2 = create_sliding_windows(X_scaled_fs2, y_scaled_fs2, lookback_window=lookback, horizon=5)
X_15min_fs2, y_15min_fs2 = create_sliding_windows(X_scaled_fs2, y_scaled_fs2, lookback_window=lookback, horizon=15)

print("\nPreparação do FS-2 concluída!")
print(f"Formato final (Shape) para horizonte de 1 min:  X={X_1min_fs2.shape}, y={y_1min_fs2.shape}")
print(f"Formato final (Shape) para horizonte de 5 min:  X={X_5min_fs2.shape}, y={y_5min_fs2.shape}")
print(f"Formato final (Shape) para horizonte de 15 min: X={X_15min_fs2.shape}, y={y_15min_fs2.shape}")

Carregando o dataset local para o FS-2...
Dataset do FS-2 filtrado com sucesso. Total de registros: 764867

Gerando janelas deslizantes para o FS-2...

Preparação do FS-2 concluída!
Formato final (Shape) para horizonte de 1 min:  X=(764807, 60, 3), y=(764807, 1)
Formato final (Shape) para horizonte de 5 min:  X=(764803, 60, 3), y=(764803, 1)
Formato final (Shape) para horizonte de 15 min: X=(764793, 60, 3), y=(764793, 1)


Validação do Pré-processamento: FS-2 (Top-3 XAI)


Os tensores para o nosso segundo cenário de testes foram gerados com sucesso, validando a etapa de redução de dimensionalidade baseada em Inteligência Artificial Explicável (XAI).Redução de Features (X): O formato tridimensional agora apresenta o shape (..., 60, 3). O valor 3 confirma que os modelos (LSTM, GRU e CNN-1D) processarão janelas de 60 minutos no passado contendo apenas as três variáveis de maior peso preditivo: clearSkyGhi, fitLuminanceCSI e whitePixelRatio.  Alvo Isolado (y): O formato se mantém em (..., 1), assegurando que a rede neural fará a predição de um único valor futuro sem vazamento de informações (data leakage).Objetivo Prático: Esta estrutura enxuta é o núcleo da nossa hipótese. Ao treinar os modelos com este dataset, poderemos comparar as métricas (RMSE, MAPE, R²) com o FS-1 e avaliar se é possível manter ou superar a acurácia original utilizando uma fração muito menor dos dados de entrada, otimizando o custo computacional do Método Híbrido de Predição (MHP).

FS-3: Top-5 XAI (Inclusão de Variáveis Intermediárias)
Variáveis Incluídas (5 variáveis):
 clearSkyGhi  
 fitLuminanceCSI  
 whitePixelRatio  
 sunLuminance  
 cloudsCoverage

O conjunto FS-3 expande o cenário anterior (FS-2) ao incorporar duas variáveis adicionais classificadas com grau de importância intermediária pela análise XAI. Enquanto a análise provou que as métricas de movimentação de nuvens (cloudsMovement) contribuem com menos de 2% cada para a predição — configurando provável ruído para a rede —, as variáveis sunLuminance e cloudsCoverage apresentam um peso analítico que pode ser benéfico. O objetivo principal ao testar o FS-3 é encontrar o "ponto de equilíbrio" na dimensionalidade dos dados: avaliar se o acréscimo destas duas informações fornece o contexto necessário para que os modelos de Deep Learning (LSTM, GRU e CNN-1D) alcancem uma acurácia superior à do FS-2, mantendo-se mais eficientes e menos custosos do que o modelo treinado com o conjunto completo (FS-1).

In [2]:
# 1. Definir o conjunto de variáveis FS-3 (Top-5 XAI)
fs3_features = [
    'clearSkyGhi',
    'fitLuminanceCSI',
    'whitePixelRatio',
    'sunLuminance',
    'cloudsCoverage'
]

# Definir a variável alvo (target)
target_col = 'clearSkyGhi'

# Juntar features e target, removendo duplicatas
cols_to_use_fs3 = list(dict.fromkeys(fs3_features + [target_col]))

# Filtrar o dataframe e remover NaNs
df_fs3 = df[cols_to_use_fs3].dropna()

print(f"Dataset do FS-3 filtrado com sucesso. Total de registros: {len(df_fs3)}")

# 2. Aplicar o MinMaxScaler exclusivo para o FS-3
scaler_X_fs3 = MinMaxScaler()
scaler_y_fs3 = MinMaxScaler()

X_scaled_fs3 = scaler_X_fs3.fit_transform(df_fs3[fs3_features])
y_scaled_fs3 = scaler_y_fs3.fit_transform(df_fs3[target_col].values.reshape(-1, 1))

# 3. Criando os datasets finais para o FS-3
lookback = 60

print("\nGerando janelas deslizantes para o FS-3...")
X_1min_fs3, y_1min_fs3 = create_sliding_windows(X_scaled_fs3, y_scaled_fs3, lookback_window=lookback, horizon=1)
X_5min_fs3, y_5min_fs3 = create_sliding_windows(X_scaled_fs3, y_scaled_fs3, lookback_window=lookback, horizon=5)
X_15min_fs3, y_15min_fs3 = create_sliding_windows(X_scaled_fs3, y_scaled_fs3, lookback_window=lookback, horizon=15)

print("\nPreparação do FS-3 concluída!")
print(f"Formato final (Shape) para horizonte de 1 min:  X={X_1min_fs3.shape}, y={y_1min_fs3.shape}")
print(f"Formato final (Shape) para horizonte de 5 min:  X={X_5min_fs3.shape}, y={y_5min_fs3.shape}")
print(f"Formato final (Shape) para horizonte de 15 min: X={X_15min_fs3.shape}, y={y_15min_fs3.shape}")

Dataset do FS-3 filtrado com sucesso. Total de registros: 764867

Gerando janelas deslizantes para o FS-3...

Preparação do FS-3 concluída!
Formato final (Shape) para horizonte de 1 min:  X=(764807, 60, 5), y=(764807, 1)
Formato final (Shape) para horizonte de 5 min:  X=(764803, 60, 5), y=(764803, 1)
Formato final (Shape) para horizonte de 15 min: X=(764793, 60, 5), y=(764793, 1)


 Validação do Pré-processamento: FS-3 (Top-5 XAI)

 A preparação dos tensores para o nosso terceiro cenário de testes (FS-3) foi concluída com sucesso, permitindo avaliar o impacto das variáveis de importância intermediária.Inclusão de Variáveis (X): O formato tridimensional apresenta o shape (..., 60, 5). O valor 5 indica que as matrizes de treinamento agora alimentam os modelos com as 3 features principais do FS-2, acrescidas de sunLuminance e cloudsCoverage.  Alvo Isolado (y): O formato (..., 1) confirma que a predição continua focada em uma única saída futura, mantendo a integridade estrutural para as redes neurais.Objetivo Prático: Este conjunto representa a busca pelo "ponto de equilíbrio" na arquitetura da rede. Ao introduzir features que a análise XAI apontou como tendo importância mediana, testamos se esse contexto adicional melhora as métricas de erro preditivo (RMSE, MAPE, R²) em comparação ao modelo extremamente enxuto (FS-2), sem trazer o ruído das variáveis de nuvem e o custo computacional do modelo completo (FS-1).  

FS-4: Sem cloudsMovement (Ablação de Variáveis de Baixo Impacto)
Variáveis Incluídas (5 variáveis):  
clearSkyGhi  
fitLuminanceCSI  
whitePixelRatio  
sunLuminance  
cloudsCoverage  

A concepção do conjunto FS-4 baseia-se na exclusão direta das variáveis que se mostraram menos relevantes na análise de Inteligência Artificial Explicável (XAI). O diagnóstico prévio do modelo MLP indicou que as variáveis relacionadas à movimentação de nuvens (cloudsMovement1, cloudsMovement5 e cloudsMovement15) apresentam uma contribuição individual inferior a 2% para a capacidade preditiva do Método Híbrido de Predição (MHP), tornando-as fortes candidatas à remoção. O objetivo do FS-4 é atuar como um teste de ablação: retirar apenas o "ruído" ou a informação de baixíssimo impacto do conjunto original (FS-1) para verificar se a ausência exclusiva das métricas de movimento de nuvens é suficiente para otimizar o treinamento e a precisão das redes neurais (LSTM, GRU e CNN-1D).

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# 1. Recarregar o dataset do zero para garantir que o 'df' existe
caminho_arquivo_csv = '/content/extracted_data/features_image_mhp.csv'
print("Carregando o dataset local...")
df = pd.read_csv(caminho_arquivo_csv, delimiter=';', decimal=',')
df.columns = df.columns.str.strip()

# 2. Definir as variáveis do FS-1 para servir de base
fs1_features = [
    'clearSkyGhi',
    'fitLuminanceCSI',
    'whitePixelRatio',
    'sunLuminance',
    'cloudsCoverage',
    'cloudsMovement1',
    'cloudsMovement5',
    'cloudsMovement15'
]

# 3. Criar o FS-4 removendo as variáveis de movimento de nuvens
fs4_features = [feature for feature in fs1_features if not feature.startswith('cloudsMovement')]
target_col = 'clearSkyGhi'

cols_to_use_fs4 = list(dict.fromkeys(fs4_features + [target_col]))
df_fs4 = df[cols_to_use_fs4].dropna()

print(f"Dataset do FS-4 filtrado com sucesso. Total de registros: {len(df_fs4)}")

# 4. Normalização
scaler_X_fs4 = MinMaxScaler()
scaler_y_fs4 = MinMaxScaler()

X_scaled_fs4 = scaler_X_fs4.fit_transform(df_fs4[fs4_features])
y_scaled_fs4 = scaler_y_fs4.fit_transform(df_fs4[target_col].values.reshape(-1, 1))

# 5. Função de Janelas Deslizantes
def create_sliding_windows(X_data, y_data, lookback_window, horizon):
    X_windows, y_windows = [], []
    for i in range(len(X_data) - lookback_window - horizon + 1):
        X_windows.append(X_data[i : (i + lookback_window)])
        y_windows.append(y_data[i + lookback_window + horizon - 1])
    return np.array(X_windows), np.array(y_windows)

lookback = 60

print("\nGerando janelas deslizantes para o FS-4...")
X_1min_fs4, y_1min_fs4 = create_sliding_windows(X_scaled_fs4, y_scaled_fs4, lookback_window=lookback, horizon=1)
X_5min_fs4, y_5min_fs4 = create_sliding_windows(X_scaled_fs4, y_scaled_fs4, lookback_window=lookback, horizon=5)
X_15min_fs4, y_15min_fs4 = create_sliding_windows(X_scaled_fs4, y_scaled_fs4, lookback_window=lookback, horizon=15)

print("\nPreparação do FS-4 concluída!")
print(f"Formato final (Shape) para horizonte de 1 min:  X={X_1min_fs4.shape}, y={y_1min_fs4.shape}")
print(f"Formato final (Shape) para horizonte de 5 min:  X={X_5min_fs4.shape}, y={y_5min_fs4.shape}")
print(f"Formato final (Shape) para horizonte de 15 min: X={X_15min_fs4.shape}, y={y_15min_fs4.shape}")

Carregando o dataset local...
Dataset do FS-4 filtrado com sucesso. Total de registros: 764867

Gerando janelas deslizantes para o FS-4...

Preparação do FS-4 concluída!
Formato final (Shape) para horizonte de 1 min:  X=(764807, 60, 5), y=(764807, 1)
Formato final (Shape) para horizonte de 5 min:  X=(764803, 60, 5), y=(764803, 1)
Formato final (Shape) para horizonte de 15 min: X=(764793, 60, 5), y=(764793, 1)


Validação do Pré-processamento: FS-4 (Ablação de Variáveis)

A preparação dos tensores para o nosso quarto e último cenário de testes (FS-4) foi concluída com sucesso. Este conjunto atua como um teste de ablação direto em relação ao modelo original.Remoção do Ruído (X): O formato tridimensional apresenta o shape (..., 60, 5). O valor 5 confirma que as três variáveis de movimentação de nuvens (cloudsMovement1, 5 e 15) foram removidas com sucesso dos dados de treinamento, restando apenas as outras cinco variáveis do conjunto completo.  Alvo Isolado (y): O formato (..., 1) assegura que a predição da rede neural se manterá focada em uma única saída futura, sem chance de vazamento de dados.Objetivo Prático: O FS-4 serve para testar a premissa de exclusão. Como a análise XAI demonstrou que as variáveis de movimento de nuvem contribuem com menos de 2% para a predição, este dataset nos permitirá verificar se apenas retirar essas variáveis de baixo impacto já é o suficiente para otimizar o treinamento e melhorar as métricas de erro dos modelos (LSTM, GRU e CNN-1D) em relação ao baseline (FS-1).

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input

def build_mlp_model(input_shape):
    """
    Constrói a arquitetura do modelo MLP de referência.
    input_shape: formato das janelas deslizantes (ex: 60 passos, 8 features para o FS-1)
    """
    model = Sequential([
        Input(shape=input_shape), # Usar Input layer como primeira camada para definir o shape
        Flatten(),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1, activation='linear') # Saída única para a predição da irradiância
    ])

    # Compilando o modelo com otimizador e função de perda
    model.compile(optimizer='adam', loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError()])

    return model

# Exemplo de inicialização usando o shape do FS-1 gerado anteriormente
# model_mlp_1min = build_mlp_model(input_shape=(lookback, len(fs1_features)))

In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import math
from sklearn.preprocessing import MinMaxScaler # Adicionar import para MinMaxScaler
import numpy as np # Adicionar import para numpy
import pandas as pd # Adicionar import para pandas

# --- Início da correção: Garantir que as variáveis do FS-1 estejam definidas ---

# Carregar o dataset do zero para garantir que o 'df' existe
caminho_arquivo_csv = '/content/extracted_data/features_image_mhp.csv'
print("Carregando o dataset local...")
df = pd.read_csv(caminho_arquivo_csv, delimiter=';', decimal=',')
df.columns = df.columns.str.strip()

# Redefinir a variável alvo (target)
target_col = 'clearSkyGhi'

# Definir as variáveis do FS-1 (garantindo que estejam no escopo)
fs1_features = [
    'clearSkyGhi',
    'fitLuminanceCSI',
    'whitePixelRatio',
    'sunLuminance',
    'cloudsCoverage',
    'cloudsMovement1',
    'cloudsMovement5',
    'cloudsMovement15'
]

# Juntar features e target, mas remover qualquer duplicata usando dict.fromkeys()
cols_to_use = list(dict.fromkeys(fs1_features + [target_col]))

# Filtrar o dataframe apenas com as colunas necessárias e remover linhas com dados faltantes (NaN)
df_fs1 = df[cols_to_use].dropna()

# Aplicar o MinMaxScaler
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler() # Este é crucial para evaluate_model

X_scaled = scaler_X.fit_transform(df_fs1[fs1_features])
y_scaled = scaler_y.fit_transform(df_fs1[target_col].values.reshape(-1, 1))

# 5. Função para gerar as Janelas Deslizantes (repetida aqui para garantir que esteja no escopo)
def create_sliding_windows(X_data, y_data, lookback_window, horizon):
    X_windows, y_windows = [], []
    for i in range(len(X_data) - lookback_window - horizon + 1):
        X_windows.append(X_data[i : (i + lookback_window)])
        y_windows.append(y_data[i + lookback_window + horizon - 1])
    return np.array(X_windows), np.array(y_windows)

lookback = 60 # Definir lookback (garantindo que esteja no escopo)

# Criando os datasets finais para os experimentos (apenas o necessário para este treino)
X_1min, y_1min = create_sliding_windows(X_scaled, y_scaled, lookback_window=lookback, horizon=1)
# --- Fim da correção ---

# 1. Função para calcular e imprimir as métricas exigidas
def evaluate_model(y_true, y_pred, model_name, horizon):
    # Desnormalizar os dados para obter o erro na escala real da irradiância
    y_true_real = scaler_y.inverse_transform(y_true)
    y_pred_real = scaler_y.inverse_transform(y_pred)

    # Calcular as métricas
    rmse = math.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mape = mean_absolute_percentage_error(y_true_real, y_pred_real) * 100 # Em porcentagem
    r2 = r2_score(y_true_real, y_pred_real)

    print(f"--- Resultados para {model_name} | Horizonte: {horizon} min ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.4f}%")
    print(f"R²:   {r2:.4f}\n")

    return rmse, mape, r2

# 2. Rotina de Treinamento e Teste (Exemplo para o horizonte de 1 minuto)
print("Iniciando o treinamento do modelo MLP de referência (FS-1 / 1 minuto)...")

# Construindo o modelo (usando a função que criamos no passo anterior)
# O input_shape é (60, 8) -> 60 passos de lookback e 8 features do FS-1
model_mlp_1min = build_mlp_model(input_shape=(lookback, len(fs1_features)))

# Treinando o modelo
# Dividimos os dados em 80% treino e 20% validação diretamente no 'fit'
history = model_mlp_1min.fit(
    X_1min, y_1min,
    epochs=50,          # Número de passadas completas no dataset
    batch_size=32,      # Tamanho do lote processado por vez
    validation_split=0.2, # Separa 20% para validação
    verbose=1           # Mostra o progresso na tela
)

print("\nTreinamento concluído. Realizando predições...")

# 3. Fazendo predições e avaliando
y_pred_1min = model_mlp_1min.predict(X_1min)
rmse_1, mape_1, r2_1 = evaluate_model(y_1min, y_pred_1min, "MLP (Ref)", 1)


Carregando o dataset local...
Iniciando o treinamento do modelo MLP de referência (FS-1 / 1 minuto)...
Epoch 1/50
19121/19121 ━━━━━━━━━━━━━━━━━━━━ 34s 2ms/step - loss: 1.5859e-04 - root_mean_squared_error: 0.0126 - val_loss: 2.6840e-05 - val_root_mean_squared_error: 0.0052
Epoch 2/50
19121/19121 ━━━━━━━━━━━━━━━━━━━━ 33s 2ms/step - loss: 1.6907e-05 - root_mean_squared_error: 0.0041 - val_loss: 7.2280e-06 - val_root_mean_squared_error: 0.0027
Epoch 3/50
19121/19121 ━━━━━━━━━━━━━━━━━━━━ 33s 2ms/step - loss: 1.1294e-05 - root_mean_squared_error: 0.0034 - val_loss: 3.3432e-06 - val_root_mean_squared_error: 0.0018
Epoch 4/50
19121/19121 ━━━━━━━━━━━━━━━━━━━━ 41s 2ms/step - loss: 9.1424e-06 - root_mean_squared_error: 0.0030 - val_loss: 3.4673e-06 - val_root_mean_squared_error: 0.0019
Epoch 5/50
19121/19121 ━━━━━━━━━━━━━━━━━━━━ 37s 2ms/step - loss: 7.4417e-06 - root_mean_squared_error: 0.0027 - val_loss: 3.5383e-06 - val_root_mean_squared_error: 0.0019
Epoch 6/50
19121/19121 ━━━━━━━━━━━━━━━━━━━